In [1]:
# Cell 1 → Imports
# Cell 2 → create_features() function
# Cell 3 → create_zone_features() function
# Cell 4 → Load one zone + build features
# Cell 5 → Train/test split
# Cell 6 → Random Forest — train + evaluate
# Cell 7 → XGBoost — train + evaluate
# Cell 8 → LightGBM — train + evaluate
# Cell 9 → Compare all 3 → winner per variable
# Cell 10 → Save best models

# Imports

In [2]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from database import load_zone_data


print('ALL LIBRARIES IMPORTED SUCCESSFULLY')


ALL LIBRARIES IMPORTED SUCCESSFULLY


In [3]:
# creating features

def create_features(df, lag_days = 14) :

    features =  pd.DataFrame(index = df.index)

    for key in ['temperature','precipitation','windspeed','humidity']:
        for j in range(1,15):
            features[f'{key}_lag{j}'] = df[key].shift(j)
        
        features[f'{key}_rolling7_mean'] = df[key].shift(1).rolling(7).mean()
        features[f'{key}_rolling14_mean'] = df[key].shift(1).rolling(14).mean()
        features[f'{key}_rolling7_std'] = df[key].shift(1).rolling(7).std()

    features['month']  = df.index.month
    features['dayofyear']  = df.index.dayofyear
    features['dayofweek']   = df.index.dayofweek


    features['latitude'] = df['latitude']
    features['longitude'] = df['longitude']
    features['altitude'] = df['altitude'] 

    return features


In [4]:
# creating zone features

def create_zone_features(zone_df, lag_days = 14):

    all_features = []
    all_targets  = []

    for city in zone_df['city'].unique():

        city_df = zone_df[zone_df['city']  ==  city].copy()
        city_df = city_df.set_index('date')
        city_df = city_df.asfreq('D')
        city_df = city_df.fillna(method = 'ffill')
        
        feat = create_features(city_df)

        feat  = feat.dropna()
        targ = city_df.loc[feat.index][['temperature',
                                 'precipitation',
                                 'windspeed',
                                 'humidity']]

        all_features.append(feat)
        all_targets.append(targ)
    
    X = pd.concat(all_features)
    y = pd.concat(all_targets)
    return X , y

In [5]:
# Cell 4 — load zone and build features
zone = 'zone2_coastal_west'
zone_df = load_zone_data(zone)

print(f"Cities: {zone_df['city'].unique()}")
print(f"Raw shape: {zone_df.shape}")

X, y = create_zone_features(zone_df)

print(f"Feature matrix X: {X.shape}")
print(f"Target matrix y:  {y.shape}")
print(f"Expected X columns: 74")

Cities: ['Goa' 'Kochi' 'Mangalore' 'Mumbai']
Raw shape: (53056, 9)
Feature matrix X: (53000, 74)
Target matrix y:  (53000, 4)
Expected X columns: 74


In [14]:
zone_df.head()

,date,temperature,precipitation,windspeed,humidity,latitude,longitude,altitude,city
0,1990-01-01,25.8,0.3,12.6,91.0,15.5,73.8,7.0,Goa
1,1990-01-02,26.0,0.0,13.2,86.0,15.5,73.8,7.0,Goa
2,1990-01-03,25.4,0.0,10.0,90.0,15.5,73.8,7.0,Goa
3,1990-01-04,26.7,0.0,12.6,84.0,15.5,73.8,7.0,Goa
4,1990-01-05,26.7,0.0,11.2,84.0,15.5,73.8,7.0,Goa


In [17]:
X.head()

,temperature_lag1,temperature_lag2,temperature_lag3,temperature_lag4,temperature_lag5,temperature_lag6,temperature_lag7,temperature_lag8,temperature_lag9,temperature_lag10,...,humidity_lag14,humidity_rolling7_mean,humidity_rolling14_mean,humidity_rolling7_std,month,dayofyear,dayofweek,latitude,longitude,altitude
date,,,,,,,,,,,,,,,,,,,,,
1990-01-15,24.4,25.4,26.3,26.7,26.6,27.8,26.9,26.7,26.3,26.7,...,91.0,70.142857,75.571429,11.334734,1,15,0,15.5,73.8,7.0
1990-01-16,24.3,24.4,25.4,26.3,26.7,26.6,27.8,26.9,26.7,26.3,...,86.0,75.000000,74.785714,4.472136,1,16,1,15.5,73.8,7.0
1990-01-17,25.6,24.3,24.4,25.4,26.3,26.7,26.6,27.8,26.9,26.7,...,90.0,74.714286,73.642857,4.750940,1,17,2,15.5,73.8,7.0
1990-01-18,24.5,25.6,24.3,24.4,25.4,26.3,26.7,26.6,27.8,26.9,...,84.0,77.428571,73.857143,8.343803,1,18,3,15.5,73.8,7.0
1990-01-19,24.0,24.5,25.6,24.3,24.4,25.4,26.3,26.7,26.6,27.8,...,84.0,80.714286,74.857143,11.250397,1,19,4,15.5,73.8,7.0


In [15]:
#  train test split
train_size = int(len(X) * 0.80)

X_train = X.iloc[:train_size]
X_test  = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test  = y.iloc[train_size:]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (42400, 74)
X_test:  (10600, 74)
y_train: (42400, 4)
y_test:  (10600, 4)


In [7]:
from sklearn.metrics import mean_squared_error
import numpy as np

def evaluate(y_true , y_pred, variable):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f'{variable:<15} RMSE: {rmse:.3f}  MAE: {mae:.3f}')

    return rmse


rf_models = {}
rf_rmse = {}

print('Random Forest - zone2_coastal_west')
print("-"* 45)



for variable in ['temperature', 'precipitation', 'windspeed', 'humidity'] :

    rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train[variable])

    y_pred = rf.predict(X_test)

    rmse = evaluate(y_test[variable], y_pred, variable)

    rf_models[variable] = rf
    rf_rmse[variable] = rmse





Random Forest - zone2_coastal_west
---------------------------------------------
temperature     RMSE: 0.676  MAE: 0.491
precipitation   RMSE: 9.870  MAE: 3.929
windspeed       RMSE: 2.703  MAE: 1.924
humidity        RMSE: 6.402  MAE: 4.202


In [9]:
# XGBoost 
xgb_models = {}
xgb_rmse = {}

print('XGBoost - zone2_coastal_west')
print('-'*45)

for variable in ['temperature', 'precipitation', 'windspeed', 'humidity'] :

    xgb = XGBRegressor(n_estimators = 100 , random_state = 42 , verbosity=0) 

    xgb.fit(X_train, y_train[variable])

    y_pred = xgb.predict(X_test)

    rmse = evaluate(y_test[variable], y_pred, variable)

    xgb_models[variable] = xgb
    xgb_rmse[variable] = rmse

    

XGBoost - zone2_coastal_west
---------------------------------------------
temperature     RMSE: 0.711  MAE: 0.509
precipitation   RMSE: 10.378  MAE: 4.045
windspeed       RMSE: 2.793  MAE: 2.029
humidity        RMSE: 6.568  MAE: 4.283


In [11]:
#   LightGBM
lgbm_models = {}
lgbm_rmse   = {}

print('LightGBM - zone2_coastal_west')
print("-" * 45)

for variable in ['temperature', 'precipitation', 'windspeed', 'humidity'] :

    lgb = LGBMRegressor(n_estimators = 100 , random_state = 42 , verbosity=-1) 

    lgb.fit(X_train, y_train[variable])

    y_pred = lgb.predict(X_test)

    rmse = evaluate(y_test[variable], y_pred, variable)

    lgbm_models[variable] = lgb
    lgbm_rmse[variable] = rmse

LightGBM - zone2_coastal_west
---------------------------------------------
temperature     RMSE: 0.661  MAE: 0.480
precipitation   RMSE: 9.530  MAE: 3.408
windspeed       RMSE: 2.582  MAE: 1.868
humidity        RMSE: 6.202  MAE: 3.979


In [12]:
# Compare all models, pick winner per variable
print("=" * 60)
print(f"{'Model Comparison — zone2_coastal_west':^60}")
print("=" * 60)
print(f"{'Variable':<15} {'RF':>8} {'XGB':>8} "
      f"{'LightGBM':>10} {'Winner':>10}")
print("-" * 60)

winners = {}

for variable in ['temperature', 'precipitation',
                 'windspeed', 'humidity']:
    
    scores = {
        'RF':       rf_rmse[variable],
        'XGBoost':  xgb_rmse[variable],
        'LightGBM': lgbm_rmse[variable]
    }
    
    winner = min(scores, key=lambda x: scores[x])
    winners[variable] = winner
    
    print(f"{variable:<15} "
          f"{rf_rmse[variable]:>8.3f} "
          f"{xgb_rmse[variable]:>8.3f} "
          f"{lgbm_rmse[variable]:>10.3f} "
          f"{winner:>10}")

print("=" * 60)
print("\nWinners per variable:")
for var, win in winners.items():
    print(f"  {var:<15} → {win}")

           Model Comparison — zone2_coastal_west            
Variable              RF      XGB   LightGBM     Winner
------------------------------------------------------------
temperature        0.676    0.711      0.661   LightGBM
precipitation      9.870   10.378      9.530   LightGBM
windspeed          2.703    2.793      2.582   LightGBM
humidity           6.402    6.568      6.202   LightGBM

Winners per variable:
  temperature     → LightGBM
  precipitation   → LightGBM
  windspeed       → LightGBM
  humidity        → LightGBM


In [13]:
#  Save winners
import os
import joblib

save_dir = f'../saved_models/zone2_coastal_west'
os.makedirs(save_dir, exist_ok=True)

model_map = {
    'RF':       rf_models,
    'XGBoost':  xgb_models,
    'LightGBM': lgbm_models
}

for variable, winner_name in winners.items():
    
    # get winning model object
    winning_model = model_map[winner_name][variable]
    
    # save to file
    path = f'{save_dir}/{variable}.pkl'
    joblib.dump(winning_model, path)
    
    print(f"Saved {winner_name} → {path}")

Saved LightGBM → ../saved_models/zone2_coastal_west/temperature.pkl
Saved LightGBM → ../saved_models/zone2_coastal_west/precipitation.pkl
Saved LightGBM → ../saved_models/zone2_coastal_west/windspeed.pkl
Saved LightGBM → ../saved_models/zone2_coastal_west/humidity.pkl


## ML Models — Key Findings (zone2_coastal_west)

### Results Summary
| Variable      | VAR RMSE | RF RMSE | XGB RMSE | LightGBM RMSE | Winner   |
|---------------|----------|---------|----------|---------------|----------|
| Temperature   | 2.013    | 0.676   | 0.711    | 0.661         | LightGBM |
| Precipitation | 16.737   | 9.870   | 10.378   | 9.530         | LightGBM |
| Windspeed     | 5.159    | 2.703   | 2.793    | 2.582         | LightGBM |
| Humidity      | 11.377   | 6.402   | 6.568    | 6.202         | LightGBM |

### Key Findings

**1. ML massively outperforms VAR**
Temperature improved 70% — from 2.013 to 0.661.
Feature engineering converts time series into tabular
format that tree models understand deeply.

**2. LightGBM wins on all variables**
Leaf-wise growth + gradient boosting gives LightGBM
edge over RF and XGBoost on large zone datasets.

**3. Precipitation still hardest variable**
RMSE 9.530 — improved 43% over VAR but still high.
Monsoon spikes need deep learning to capture properly.

**4. Decision for production pipeline**
LightGBM → primary ML model in src/model_ml.py
RF and XGBoost → trained as challengers in model_selector.py

### Baseline scores for LSTM to beat
- Temperature   RMSE < 0.661
- Precipitation RMSE < 9.530
- Windspeed     RMSE < 2.582
- Humidity      RMSE < 6.202